# Assignment 3: Text Preprocessing and Classification

In [1]:
%pip install pandas numpy nltk scikit-learn

Note: you may need to restart the kernel to use updated packages.


Import required libraries

In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import nltk
import zipfile
import urllib.request
import os

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Step 1: Download required NLTK resources

In [16]:
# --------------------------------------------------
# Step 1: Download required NLTK resources
# --------------------------------------------------

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\yashr\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yashr\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yashr\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yashr\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\yashr\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\yashr\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-d

True

# Step 2: Load the SMS Spam Collection dataset

In [17]:
# --------------------------------------------------
# Step 2: Load the SMS Spam Collection dataset
# --------------------------------------------------

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam.zip"

# Download dataset if it is not already available
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)

# Extract dataset
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("sms_dataset")

# Read the dataset
file_path = "sms_dataset/SMSSpamCollection"

df = pd.read_csv(
    file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

print("Dataset loaded successfully!")
print(df.head())

print("\nDataset shape:", df.shape)
print("\nClass distribution:")
print(df['label'].value_counts())

Dataset loaded successfully!
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Dataset shape: (5572, 2)

Class distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


# Step 3: Text preprocessing

In [18]:
# --------------------------------------------------
# Step 3: Text preprocessing
# --------------------------------------------------

# Initialize preprocessing tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Function to clean text
def clean_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize text into words
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # Apply stemming
    stemmed_tokens = [
        stemmer.stem(word) for word in tokens
    ]

    # Apply lemmatization
    lemmatized_tokens = [
        lemmatizer.lemmatize(word) for word in stemmed_tokens
    ]

    # Return processed text
    return ' '.join(lemmatized_tokens)


# Apply preprocessing to all messages
df['processed_message'] = df['message'].apply(clean_text)

print("\nOriginal and preprocessed text:")
print(df[['message', 'processed_message']].head())



Original and preprocessed text:
                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                   processed_message  
0  go jurong point crazi avail bugi n great world...  
1                              ok lar joke wif u oni  
2  free entri wkli comp win fa cup final tkt st m...  
3                u dun say earli hor u c alreadi say  
4          nah dont think goe usf live around though  


# Step 4: POS (Part-of-Speech) tagging

In [19]:

# --------------------------------------------------
# Step 4: POS (Part-of-Speech) tagging
# --------------------------------------------------

# POS tagging identifies grammatical categories
# such as nouns, verbs, adjectives, and adverbs.

def perform_pos_tagging(text):

    tokens = word_tokenize(text)

    tagged_tokens = pos_tag(tokens)

    return tagged_tokens

# Apply POS tagging to processed messages
df['POS_tags'] = df['processed_message'].apply(
    perform_pos_tagging
)

print("\nPOS tagging examples:")
print(df[['processed_message', 'POS_tags']].head())


POS tagging examples:
                                   processed_message  \
0  go jurong point crazi avail bugi n great world...   
1                              ok lar joke wif u oni   
2  free entri wkli comp win fa cup final tkt st m...   
3                u dun say earli hor u c alreadi say   
4          nah dont think goe usf live around though   

                                            POS_tags  
0  [(go, VB), (jurong, JJ), (point, NN), (crazi, ...  
1  [(ok, JJ), (lar, JJ), (joke, NN), (wif, NN), (...  
2  [(free, JJ), (entri, NN), (wkli, NN), (comp, N...  
3  [(u, JJ), (dun, NNS), (say, VBP), (earli, JJ),...  
4  [(nah, JJ), (dont, NN), (think, VBP), (goe, JJ...  


# Step 5: One-hot encoding of class labels

In [20]:

# --------------------------------------------------
# Step 5: One-hot encoding of class labels
# --------------------------------------------------

# Convert spam and ham labels into numerical
# one-hot encoded representations.

encoder = OneHotEncoder(sparse_output=False)

label_encoded = encoder.fit_transform(
    df[['label']]
)

print("\nOne-hot encoded labels:")
print(pd.DataFrame(
    label_encoded,
    columns=encoder.get_feature_names_out(['label'])
).head())



One-hot encoded labels:
   label_ham  label_spam
0        1.0         0.0
1        1.0         0.0
2        0.0         1.0
3        1.0         0.0
4        1.0         0.0


# Step 6: Bag of Words and N-gram feature extraction

In [21]:

# --------------------------------------------------
# Step 6: Bag of Words and N-gram feature extraction
# --------------------------------------------------

# Bag of Words with unigram features
bow_vectorizer = CountVectorizer(
    ngram_range=(1, 1)
)

X_bow = bow_vectorizer.fit_transform(
    df['processed_message']
)

print("\nBag of Words shape:", X_bow.shape)

# Unigram and bigram features
bigram_vectorizer = CountVectorizer(
    ngram_range=(1, 2)
)

X_bigram = bigram_vectorizer.fit_transform(
    df['processed_message']
)

print("Unigram + Bigram shape:", X_bigram.shape)

# N-gram features up to trigram
ngram_vectorizer = CountVectorizer(
    ngram_range=(1, 3)
)

X_ngram = ngram_vectorizer.fit_transform(
    df['processed_message']
)

print("Unigram + Bigram + Trigram shape:", X_ngram.shape)



Bag of Words shape: (5572, 7055)
Unigram + Bigram shape: (5572, 36874)
Unigram + Bigram + Trigram shape: (5572, 67083)


# Step 7: TF-IDF feature extraction

In [22]:

# --------------------------------------------------
# Step 7: TF-IDF feature extraction
# --------------------------------------------------

# TF-IDF assigns importance to words based on
# their frequency within messages and across documents.

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000
)

X_tfidf = tfidf_vectorizer.fit_transform(
    df['processed_message']
)

print("\nTF-IDF feature shape:", X_tfidf.shape)

# Display sample TF-IDF feature names
print("\nSample TF-IDF features:")
print(tfidf_vectorizer.get_feature_names_out()[:20])


TF-IDF feature shape: (5572, 5000)

Sample TF-IDF features:
['aah' 'aathilov' 'aathilov lot' 'aathiwher' 'aathiwher dear' 'abi'
 'abiola' 'abl' 'abl get' 'abt' 'abta' 'abta complimentari' 'ac' 'acc'
 'accept' 'accept brother' 'accept dayu' 'access' 'accid' 'accident']


# Step 8: Prepare data for classification

In [23]:
# --------------------------------------------------
# Step 8: Prepare data for classification
# --------------------------------------------------

X = df['processed_message']
y = df['label']

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 4457
Testing samples: 1115


# Step 9: Convert text into TF-IDF vectors

In [24]:
# --------------------------------------------------
# Step 9: Convert text into TF-IDF vectors
# --------------------------------------------------

# Fit vectorizer only on training data to avoid
# data leakage from the test set.

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("\nTraining TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)


Training TF-IDF shape: (4457, 5000)
Testing TF-IDF shape: (1115, 5000)


# Step 10: Train text classification model

In [25]:
# --------------------------------------------------
# Step 10: Train text classification model
# --------------------------------------------------

# Multinomial Naive Bayes is suitable for text
# classification using word-frequency features.

model = MultinomialNB()

# Train model
model.fit(X_train_tfidf, y_train)

print("\nModel trained successfully!")


Model trained successfully!


# Step 11: Make predictions

In [26]:
# --------------------------------------------------
# Step 11: Make predictions
# --------------------------------------------------

y_pred = model.predict(X_test_tfidf)

# Step 12: Evaluate the classification model

In [27]:
# --------------------------------------------------
# Step 12: Evaluate the classification model
# --------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    pos_label='spam'
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label='spam'
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label='spam'
)

print("\n========== MODEL EVALUATION ==========")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


========== MODEL EVALUATION ==========
Accuracy  : 0.9704
Precision : 1.0000
Recall    : 0.7785
F1-Score  : 0.8755

Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.78      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115


Confusion Matrix:
[[966   0]
 [ 33 116]]


# Step 13: Test the model on a new message

In [28]:


# --------------------------------------------------
# Step 13: Test the model on a new message
# --------------------------------------------------

new_message = [
    "Congratulations! You have won a free prize. Claim now!"
]

# Preprocess new message
processed_new_message = [
    clean_text(message) for message in new_message
]

# Convert message into TF-IDF features
new_message_tfidf = tfidf_vectorizer.transform(
    processed_new_message
)

# Predict message category
prediction = model.predict(new_message_tfidf)

print("\nNew Message:", new_message[0])
print("Predicted Category:", prediction[0])

if prediction[0] == 'spam':
    print("The message is classified as SPAM.")
else:
    print("The message is classified as HAM (Not Spam).")

print("\nAssignment completed successfully!")


New Message: Congratulations! You have won a free prize. Claim now!
Predicted Category: spam
The message is classified as SPAM.

Assignment completed successfully!
